# Smoke Tests

I run this notebook top-to-bottom to verify the data layer is healthy.
Each section is independent after section 1. If something fails, I check [docs/HOW_TO_RUN_TESTS.md](../docs/HOW_TO_RUN_TESTS.md).

## 1. Setup — imports + schema check

In [ ]:
from pathlib import Path
import sys

# Find project root regardless of where Jupyter started
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.common.database import DatabaseManager
from src.common.env_loader import get_api_key
from src.common.datasources.finviz_source import FinvizSource
from src.common.datasources.yahoo_source import YahooSource

_results = {}  # tally each section's pass/fail for the final summary

try:
    db = DatabaseManager()
    db.migrate_to_v4()
    version = db.get_schema_version()
    assert version == 4, f'expected schema v4, got {version}'
    print(f'✓ Imports OK')
    print(f'✓ Database at schema v{version}')
    print(f'✓ Project root: {ROOT}')
    _results['setup'] = 'PASS'
except Exception as e:
    print(f'✗ FAILED: {type(e).__name__}: {e}')
    _results['setup'] = 'FAIL'
    raise

## 2. API key check — live calls to FMP / Polygon / Tiingo

Validates that the API keys in `.env` actually work. Takes ~10 seconds (one call per provider, rate-polite).

In [ ]:
import urllib.request, urllib.error, time

# Each test hits a known free-tier endpoint per provider.
# FMP uses /stable/ (post-2024 API); legacy /api/v3/ returns 403 for new keys.
tests = [
    ('fmp',     'https://financialmodelingprep.com/stable/profile?symbol=AAPL&apikey={k}'),
    ('polygon', 'https://api.polygon.io/v3/reference/tickers/AAPL?apiKey={k}'),
    ('tiingo',  'https://api.tiingo.com/api/test?token={k}'),
]

key_ok = 0
for provider, url in tests:
    k = get_api_key(provider)
    if not k:
        print(f'  {provider:>8}: ✗ NOT SET in .env (skipped)')
        continue
    masked = k[:4] + '*'*(len(k)-8) + k[-4:] if len(k) > 8 else '****'
    try:
        req = urllib.request.Request(url.format(k=k), headers={'User-Agent': 'TradeIdentifier/0.1'})
        with urllib.request.urlopen(req, timeout=10) as r:
            print(f'  {provider:>8}: ✓ OK  (HTTP {r.status}, key={masked}, {len(r.read())} bytes returned)')
            key_ok += 1
    except urllib.error.HTTPError as e:
        print(f'  {provider:>8}: ✗ HTTP {e.code} {e.reason} (key={masked})')
    except Exception as e:
        print(f'  {provider:>8}: ✗ {type(e).__name__}: {e}')
    time.sleep(1)

_results['api_keys'] = 'PASS' if key_ok >= 1 else 'FAIL'
print(f"\n→ {key_ok}/3 keys valid")

## 3. Automated unit + module test suite

Runs ~177 pytest tests against fixture data. ~15 seconds.

In [ ]:
import subprocess

py = ROOT / 'venv' / 'Scripts' / 'python.exe'
if not py.exists():
    py = sys.executable  # fallback if venv layout differs

result = subprocess.run(
    [str(py), '-m', 'pytest', '-m', 'not integration', '-q', '--no-header'],
    capture_output=True, text=True, cwd=str(ROOT), timeout=120,
)
# Show last 15 lines (the summary)
tail = '\n'.join(result.stdout.strip().split('\n')[-15:])
print(tail)
if 'passed' in result.stdout and 'failed' not in result.stdout:
    print('\n✓ All pytest checks pass')
    _results['pytest'] = 'PASS'
else:
    print('\n✗ pytest reports failures — see output above')
    _results['pytest'] = 'FAIL'

## 4. Live Finviz universe pull

Pulls the configured universe band (default `+Mid (over $2bln)` per `config/datasources.yaml`).
Returns ~2,500 US mid-cap-and-above stocks. ~5 seconds.

In [ ]:
import datetime
run_id = f'smoke-{datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%S")}'
try:
    finviz_df = FinvizSource().fetch_universe(run_id=run_id)
    print(f'✓ Got {len(finviz_df)} rows from Finviz')
    print(f'   columns: {list(finviz_df.columns)[:10]}...')
    display(finviz_df[['Ticker','Company','Sector','Market Cap','P/E']].head(8))
    _results['finviz'] = 'PASS'
except Exception as e:
    print(f'✗ Finviz FAILED: {type(e).__name__}: {e}')
    _results['finviz'] = 'FAIL'
    finviz_df = pd.DataFrame()

## 5. Live Yahoo fundamentals — single ticker

Pulls AAPL's current fundamentals from Yahoo. ~3 seconds.

In [ ]:
yahoo = YahooSource()
try:
    row = yahoo.fetch_fundamentals_for_ticker('AAPL', run_id=run_id)
    if row is None:
        print('✗ Yahoo returned None (possible throttle or empty .info)')
        _results['yahoo_fundamentals'] = 'FAIL'
    else:
        d = row.model_dump()
        for k, v in d.items():
            print(f'  {k:<28} {v}')
        print(f'\n✓ Yahoo fundamentals OK for AAPL')
        _results['yahoo_fundamentals'] = 'PASS'
except Exception as e:
    print(f'✗ Yahoo FAILED: {type(e).__name__}: {e}')
    _results['yahoo_fundamentals'] = 'FAIL'

## 6. Live Yahoo historical prices + watermark verification

Pulls AAPL's price history, observes watermark, then re-pulls — the second call should fetch ZERO new rows (proof Principle 6 works). ~5 seconds total.

In [ ]:
try:
    n_first = yahoo.fetch_historical_price('AAPL', db=db)
    w = db.get_watermark('yahoo', 'AAPL', 'historical_price')
    print(f'✓ First fetch: inserted {n_first} rows')
    print(f'   watermark: last_observation_date = {w["last_observation_date"]}')
    print(f'   watermark: fetch_count = {w["fetch_count"]}, error_count = {w["error_count"]}')

    n_second = yahoo.fetch_historical_price('AAPL', db=db)
    print(f'\n✓ Second fetch: inserted {n_second} rows  (should be 0 if today is the same trading day)')
    if n_second == 0:
        print('   → Principle 6 confirmed: no redundant fetches')
    _results['watermarks'] = 'PASS'
except Exception as e:
    print(f'✗ FAILED: {type(e).__name__}: {e}')
    _results['watermarks'] = 'FAIL'

## 7. Database inspection

Show what's actually in the persistent storage right now.

In [ ]:
import sqlite3
with sqlite3.connect(db.db_path) as conn:
    tables = pd.read_sql(
        "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%' ORDER BY name",
        conn,
    )
    print('Tables + views in data/fundamentals.db:')
    display(tables)

    print('\nfetch_watermarks (what data we already have):')
    wm = pd.read_sql(
        'SELECT source, ticker, field, last_observation_date, fetch_count, error_count FROM fetch_watermarks ORDER BY source, ticker',
        conn,
    )
    if len(wm):
        display(wm.head(20))
    else:
        print('  (none yet — run Sections 4-6 first)')

    print('\nhistorical_price row counts per ticker:')
    cnts = pd.read_sql(
        'SELECT ticker, COUNT(*) AS n_rows, MIN(observation_date) AS earliest, MAX(observation_date) AS latest FROM historical_price GROUP BY ticker',
        conn,
    )
    if len(cnts):
        display(cnts)
    else:
        print('  (empty)')
_results['db_inspect'] = 'PASS'

## 8. Summary

In [ ]:
print('=' * 60)
print('SMOKE TEST RESULTS')
print('=' * 60)
for section, result in _results.items():
    icon = '✓' if result == 'PASS' else '✗'
    print(f'  {icon} {section:<25} {result}')

n_pass = sum(1 for v in _results.values() if v == 'PASS')
n_total = len(_results)
print()
if n_pass == n_total:
    print(f'ALL SMOKE TESTS PASSED ({n_pass}/{n_total})')
    print('System is healthy. Safe to continue building.')
else:
    print(f'{n_total - n_pass} OF {n_total} SECTIONS FAILED')
    print('See docs/HOW_TO_RUN_TESTS.md for what to do next.')